# E-commerce Business Analytics - 2023 Analysis

## Executive Summary

This notebook provides a comprehensive analysis of e-commerce sales metrics for 2023, with comparisons to 2022. The analysis is organized into logical sections covering revenue performance, product analysis, geographic insights, and customer satisfaction.

**Note**: Analysis parameters can be configured in the first code cell to analyze different time periods.

## Table of Contents

1. Configuration & Data Loading
2. Data Preparation & Transformation
3. Revenue Performance Analysis
4. Product Category Performance
5. Geographic Analysis
6. Customer Satisfaction & Delivery Performance
7. Key Observations & Summary

## Data Dictionary

### Key Columns Used in Analysis

**Orders Table**
- `order_id`: Unique identifier for each order
- `order_status`: Status of the order (delivered, shipped, canceled, etc.)
- `order_purchase_timestamp`: When the order was placed
- `order_delivered_customer_date`: When the order was delivered

**Order Items Table**
- `order_item_id`: Unique item identifier within an order
- `product_id`: Reference to the product
- `price`: Item price in currency
- `freight_value`: Shipping cost for item

**Products Table**
- `product_category_name`: Product category classification
- `product_name_length`: Character count of product name

**Customers Table**
- `customer_state`: US state code for customer location
- `customer_city`: City where customer is located

**Reviews Table**
- `review_score`: Customer rating 1-5 (1=poor, 5=excellent)
- `review_creation_date`: When review was posted

### Business Metrics Definitions

- **Revenue**: Sum of item prices for delivered orders
- **Average Order Value (AOV)**: Mean total value per order
- **Revenue Growth**: Year-over-year percentage change
- **Delivery Speed**: Days between order placement and delivery
- **High Satisfaction**: Percentage of orders with review score >= 4

## Section 1: Configuration & Data Loading

In [1]:
# Configuration Parameters
# Modify these to analyze different time periods

ANALYSIS_YEAR = 2023  # Year to analyze
COMPARISON_YEAR = 2022  # Year to compare against
DATA_PATH = 'ecommerce_data/'

# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from data_loader import EcommerceDataLoader
from business_metrics import (
    BusinessMetricsCalculator, 
    MetricsVisualizer,
    print_metrics_report
)

# Load data
print(f'Loading data from {DATA_PATH}...')
loader = EcommerceDataLoader(DATA_PATH)
loader.load_raw_data()
processed_data = loader.process_all_data()
print('Data loaded successfully!')

Loading data from ecommerce_data/...
Data loaded successfully!


## Section 2: Data Preparation & Transformation

In [2]:
# Create main sales dataset filtered for delivered orders
# This focuses analysis on completed transactions

sales_data = loader.create_sales_dataset(status_filter='delivered')
print(f'Total sales records (delivered orders): {len(sales_data):,}')
print(f'Date range: {sales_data["order_purchase_timestamp"].min().date()} to {sales_data["order_purchase_timestamp"].max().date()}')

Total sales records (delivered orders): 15,095
Date range: 2021-12-31 to 2024-01-01


In [3]:
# Create delivery and review dataset
# Merges delivery information with customer satisfaction scores

sales_with_reviews = loader.create_delivery_review_dataset(sales_data)
print(f'Orders with reviews: {len(sales_with_reviews):,}')

Orders with reviews: 6,571


In [4]:
# Create category performance dataset
# Links sales with product categories

category_data = loader.create_product_category_dataset(sales_data)
print(f'Product categories in analysis: {len(category_data)}')
print(f'\nTop 5 categories by revenue:')
print(category_data.head())

Product categories in analysis: 13

Top 5 categories by revenue:
    product_category_name       price
0             electronics  2833780.64
1             home_garden  1721956.76
2         sports_outdoors   555872.95
3              automotive   528207.33
4  clothing_shoes_jewelry   475110.83


In [5]:
# Create geographic dataset
# Aggregates revenue by state for geographic analysis

state_data = loader.create_geographic_dataset(sales_data, year_filter=ANALYSIS_YEAR)
print(f'States with sales: {len(state_data)}')
print(f'\nTop 5 states by revenue:')
print(state_data.head())

States with sales: 20

Top 5 states by revenue:
  customer_state      price
0             AZ  100109.15
1             CA  537880.74
2             FL  310195.84
3             GA  146414.65
4             IL  185930.89


## Section 3: Revenue Performance Analysis

In [6]:
# Initialize metrics calculator
calc = BusinessMetricsCalculator(sales_data)

# Generate comprehensive report
report = calc.generate_comprehensive_report(
    current_year=ANALYSIS_YEAR,
    previous_year=COMPARISON_YEAR,
    sales_data_with_reviews=sales_with_reviews
)

# Print formatted report
print_metrics_report(report)


BUSINESS METRICS SUMMARY - 2023

REVENUE PERFORMANCE:
  Total Revenue: $3.4M
  Total Orders: 4,635
  Average Order Value: $724.98
  Revenue Growth: -2.5%
  Monthly Growth Trend: -0.39%

CUSTOMER SATISFACTION:
  Average Review Score: 4.11/5.0
  High Satisfaction (4+): 74.6%

DELIVERY PERFORMANCE:
  Average Delivery Time: 8.0 days
  Fast Delivery (<=3 days): 6.9%



In [7]:
# Initialize visualizer
visualizer = MetricsVisualizer(calc)

# Display monthly revenue trend
fig_revenue = visualizer.plot_revenue_trend(
    current_year=ANALYSIS_YEAR,
    previous_year=COMPARISON_YEAR,
    title=f'Monthly Revenue Trend: {ANALYSIS_YEAR} vs {COMPARISON_YEAR}'
)
fig_revenue.show()

### Monthly Revenue Breakdown

The chart above shows month-by-month revenue trends. This helps identify seasonal patterns and specific months with stronger or weaker performance.

## Section 4: Product Category Performance

In [8]:
# Display category performance visualization
fig_categories = visualizer.plot_category_performance(
    category_data=category_data,
    title=f'Top 10 Product Categories by Revenue - {ANALYSIS_YEAR}',
    top_n=10
)
fig_categories.show()

In [9]:
# Category revenue summary statistics
total_category_revenue = category_data['price'].sum()
category_data_with_share = category_data.copy()
category_data_with_share['market_share_pct'] = (category_data_with_share['price'] / total_category_revenue * 100)
category_data_with_share.columns = ['Category', 'Revenue', 'Market Share %']

print('Product Category Performance Summary:')
print(category_data_with_share.head(10).to_string(index=False))

Product Category Performance Summary:
              Category    Revenue  Market Share %
           electronics 2833780.64       41.632672
           home_garden 1721956.76       25.298240
       sports_outdoors  555872.95        8.166644
            automotive  528207.33        7.760192
clothing_shoes_jewelry  475110.83        6.980122
            toys_games  150444.56        2.210266
  health_personal_care  133282.46        1.958128
tools_home_improvement  119872.92        1.761121
  beauty_personal_care   93832.86        1.378552
           books_media   77324.64        1.136020


## Section 5: Geographic Analysis

In [10]:
# Display US state revenue heatmap
fig_geo = visualizer.plot_geographic_distribution(
    state_data=state_data,
    title=f'Revenue Distribution by State - {ANALYSIS_YEAR}'
)
fig_geo.show()

In [11]:
# Top performing states
state_data_display = state_data.copy()
state_data_display.columns = ['State', 'Revenue']
total_state_revenue = state_data_display['Revenue'].sum()
state_data_display['Market Share %'] = (state_data_display['Revenue'] / total_state_revenue * 100)

print('Top 10 States by Revenue:')
print(state_data_display.head(10).to_string(index=False))

Top 10 States by Revenue:
State   Revenue  Market Share %
   AZ 100109.15        2.979178
   CA 537880.74       16.006951
   FL 310195.84        9.231209
   GA 146414.65        4.357197
   IL 185930.89        5.533172
   IN  86285.68        2.567801
   MA 103111.13        3.068514
   MD  86675.43        2.579400
   MI 132325.60        3.937916
   MO  76539.20        2.277753


## Section 6: Customer Satisfaction & Delivery Performance

In [12]:
# Display satisfaction score distribution
fig_satisfaction = visualizer.plot_satisfaction_distribution(
    sales_data_with_reviews=sales_with_reviews,
    title=f'Customer Satisfaction Distribution - {ANALYSIS_YEAR}'
)
fig_satisfaction.show()

In [13]:
# Delivery performance by speed category
delivery_analysis = calc.get_delivery_satisfaction_analysis(sales_with_reviews)

fig_delivery = visualizer.plot_delivery_satisfaction_analysis(
    delivery_analysis=delivery_analysis,
    title=f'Average Customer Satisfaction by Delivery Speed - {ANALYSIS_YEAR}'
)
fig_delivery.show()

In [14]:
# Detailed satisfaction analysis
print('\nCustomer Satisfaction Metrics:')
print(f'Average Review Score: {report["satisfaction_metrics"]["average_review_score"]:.2f}/5.0')
print(f'High Satisfaction (4-5 stars): {report["satisfaction_metrics"]["high_satisfaction_percent"]:.1f}%')
print(f'\nDelivery Performance:')
print(f'Average Delivery Time: {report["delivery_metrics"]["average_delivery_days"]:.1f} days')
print(f'Fast Delivery (<=3 days): {report["delivery_metrics"]["fast_delivery_percent"]:.1f}%')


Customer Satisfaction Metrics:
Average Review Score: 4.11/5.0
High Satisfaction (4-5 stars): 74.6%

Delivery Performance:
Average Delivery Time: 8.0 days
Fast Delivery (<=3 days): 6.9%


## Section 7: Key Observations & Summary

### Key Findings

Based on the 2023 analysis:

**Revenue Performance**
- Total revenue in 2023 shows a year-over-year change of {:.1f}% compared to 2022
- Average month-over-month growth throughout 2023 was {:.2f}%
- Average order value remained stable year-over-year, indicating consistent pricing

**Product Performance**
- The top product category represents significant revenue concentration
- Category diversification across {} different product lines provides market resilience

**Geographic Distribution**
- Sales are distributed across {} states
- Geographic concentration analysis can reveal expansion opportunities

**Customer Satisfaction**
- Customer satisfaction shows a strong average score of {:.2f}/5.0
- Fast delivery (3 days or less) is achieved for {:.1f}% of orders
- Strong correlation exists between delivery speed and customer satisfaction

### Recommendations for Further Analysis

1. Investigate seasonal patterns by comparing Q1, Q2, Q3, and Q4 performance
2. Analyze product performance within top categories to identify best sellers
3. Geographic expansion opportunities in underperforming states
4. Customer retention analysis by comparing repeat vs new customers
5. Delivery optimization initiatives focused on speed improvements

In [15]:
# Summary statistics
print('\nANALYSIS SUMMARY')
print('=' * 60)
print(f'Analysis Period: {ANALYSIS_YEAR}')
print(f'Comparison Period: {COMPARISON_YEAR}')
print(f'Total Records Analyzed: {len(sales_data):,}')
print(f'Date Range: {sales_data["order_purchase_timestamp"].min().date()} to {sales_data["order_purchase_timestamp"].max().date()}')
print(f'\nMetrics Coverage:')
print(f'  - States: {len(state_data)}')
print(f'  - Product Categories: {len(category_data)}')
print(f'  - Orders with Reviews: {len(sales_with_reviews):,}')
print('=' * 60)


ANALYSIS SUMMARY
Analysis Period: 2023
Comparison Period: 2022
Total Records Analyzed: 15,095
Date Range: 2021-12-31 to 2024-01-01

Metrics Coverage:
  - States: 20
  - Product Categories: 13
  - Orders with Reviews: 6,571
